# 4 · LlamaIndex — RAG sobre documentos de la Universidad
### Sesión 1 — Agentes de IA, Orquestación y Protocolos

**Complejidad: 🟡 Media-alta**   |   **Dependencias: `llama-index`, `llama-index-llms-anthropic`, `llama-index-embeddings-huggingface`**

⚠️ **Este es el notebook con la instalación más pesada** de la serie: descarga un modelo de embeddings de HuggingFace (~90 MB) además de las librerías de LlamaIndex. Si tu ambiente tiene poca memoria o mala conexión, es normal que tarde 2-3 minutos.

Vemos el pipeline **Load → Chunk → Embed → Index → Query** explicado en la charla, aplicado a un mini set de documentos de ejemplo.


## 🎯 Objetivo de aprendizaje

Al terminar este notebook vas a poder:
- Explicar qué es RAG (*Retrieval-Augmented Generation*) y qué problema resuelve.
- Describir el pipeline Load → Chunk → Embed → Index → Query.
- Construir un índice sobre documentos propios y hacer preguntas que el modelo responde citando esa información, no su conocimiento general.


## 📚 Teoría: RAG y LlamaIndex

Un LLM no conoce tus documentos privados — su conocimiento viene de datos de entrenamiento genéricos, no del reglamento de tu universidad ni de tu calendario académico. **RAG** (*Retrieval-Augmented Generation*) resuelve esto sin reentrenar el modelo: en el momento de la consulta, recuperas los fragmentos de tus documentos más relevantes para la pregunta, y se los das al modelo como contexto adicional junto con el prompt.

El pipeline típico tiene 5 pasos:
1. **Load**: cargar los documentos fuente (PDF, texto, páginas web, bases de datos).
2. **Chunk**: dividir cada documento en fragmentos manejables.
3. **Embed**: convertir cada fragmento en un vector numérico que captura su significado semántico.
4. **Index**: almacenar esos vectores en una estructura que permite búsqueda por similitud.
5. **Query**: dada una pregunta, recuperar los fragmentos más relevantes y generar la respuesta final con el modelo.

**LlamaIndex** está especializado en este pipeline: conectores nativos a muchas fuentes de datos e índices optimizados para recuperación semántica. **LangChain** (notebook anterior) es más general, pensado para orquestar flujos y agentes — de hecho, es común combinar ambos: usar LlamaIndex para la parte de recuperación de datos, y LangChain (o un agente propio) para orquestar el resto del flujo.


## 0. Instalación

In [ ]:
!pip install -q llama-index llama-index-llms-anthropic llama-index-embeddings-huggingface

### Configurar API key de Anthropic

**Cómo obtenerla:** [console.anthropic.com](https://console.anthropic.com/settings/keys)

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `ANTHROPIC_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["ANTHROPIC_API_KEY"] = os.environ.get("ANTHROPIC_API_KEY") or getpass("Pega tu ANTHROPIC_API_KEY: ")

print("API key configurada:", "OK" if os.environ.get("ANTHROPIC_API_KEY") else "FALTA")


## 1. Documentos de ejemplo

In [ ]:
import os
os.makedirs("docs_universidad", exist_ok=True)

documentos = {
    "reglamento_biblioteca.txt": """
La biblioteca central abre de lunes a viernes de 6:00am a 10:00pm y sábados de 8:00am a 4:00pm.
Los estudiantes pueden reservar salas de estudio grupal por hasta 3 horas al día, a través del portal web.
El préstamo de libros para estudiantes de pregrado es de 7 días, renovable dos veces si no hay reservas pendientes.
""",
    "calendario_academico.txt": """
El semestre 2026-2 inicia el 4 de agosto y finaliza el 5 de diciembre.
Las inscripciones de materias son del 20 al 24 de julio.
La semana de receso intersemestral es del 12 al 16 de octubre.
""",
    "proceso_grados.txt": """
Para solicitar grado, el estudiante debe tener el 100% de créditos aprobados y paz y salvo financiero.
La solicitud se hace en la Oficina de Registro Académico con al menos 30 días de anticipación a la ceremonia.
Las ceremonias de grado se realizan en marzo, julio y noviembre de cada año.
"""
}

for filename, content in documentos.items():
    with open(f"docs_universidad/{filename}", "w") as f:
        f.write(content)

print("Documentos de ejemplo creados en docs_universidad/")


## 2. Pipeline de RAG: Load → Chunk → Embed → Index → Query

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.llms.anthropic import Anthropic as LI_Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configuración global: qué LLM y qué modelo de embeddings usar
Settings.llm = LI_Anthropic(model="claude-sonnet-4-6")
Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 1. Load
docs = SimpleDirectoryReader("docs_universidad").load_data()
print(f"Documentos cargados: {len(docs)}")

# 2-4. Chunk + Embed + Index (LlamaIndex lo hace en una sola llamada)
index = VectorStoreIndex.from_documents(docs)

# 5. Query
query_engine = index.as_query_engine()
respuesta = query_engine.query("¿Cuándo son las inscripciones de materias del próximo semestre?")
print(respuesta)


## 🧪 Ejercicio

Agrega un cuarto documento (por ejemplo, sobre becas o convalidaciones) y haz una pregunta que solo se pueda responder combinando información de **dos** documentos distintos. Observa en `respuesta.source_nodes` qué fragmentos usó el modelo para responder.

---
**Siguiente notebook:** `mcp_servidor_cliente.ipynb` — servidor y cliente MCP mínimos.


In [ ]:
# Tu código aquí
